<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/01_fatura_dataset_inspection_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/01_fatura_dataset_inspection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinDocAI — FATURA2 Dataset Inspection & OCR Pipeline

Initial exploration, quality audit, split generation, and `docTR` OCR extraction for the **FATURA2** invoice dataset (10,000 invoices across 50 templates).

**Pipeline stages:**
1. **Environment & Data Setup** — install packages, fetch FATURA2 from Zenodo, and configure working paths.
2. **Dataset & Annotation Inspection** — inspect default `strat1` splits, compare all three annotation formats (`Original_Format`, `COCO_compatible_format`, `layoutlm_HF_format`), and check bounding-box coordinate orientation.
3. **Single-Image OCR Sanity Check** — run `docTR` on one sample invoice to verify detections, reading order, and text matching.
4. **OCR vs. Ground Truth Audit** — analyze per-field OCR mismatches across all 10k invoices (either by loading the saved audit CSV or re-running from scratch).
5. **Leakage-Free Template Split & Manifest Freeze** — build a strict 32/8/10 template-disjoint split with an in-layout holdout and lock the test set with a SHA-256 hash.
6. **Sharded Full-Dataset OCR Extraction** — run batched `docTR` inference with resume support and sync JSONL shards to Hugging Face Hub.

## 1. Environment Setup & Dataset Download

Install `python-doctr`, `mplcursors` (used by docTR's interactive plot), `huggingface_hub`, and `kagglehub`. Then download and extract FATURA2 from Zenodo if it isn't already on disk.

In [ ]:
!pip install -q python-doctr mplcursors huggingface_hub kagglehub

In [ ]:
import glob
import hashlib
import json
import math
import os
import random
import re
import subprocess
import time
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from doctr.io import DocumentFile
from doctr.models import ocr_predictor

In [ ]:
# Skip downloading/extracting if the dataset is already present in this session
DATA_ZIP = Path("/content/FATURA2.zip")
EXTRACT_DIR = Path("/content/FATURA")
BASE_DIR = EXTRACT_DIR / "invoices_dataset_final"

if not BASE_DIR.exists():
    subprocess.run(
        [
            "wget", "-c",
            "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1",
            "-O", str(DATA_ZIP),
        ],
        check=True,
    )
    subprocess.run(["unzip", "-q", str(DATA_ZIP), "-d", str(EXTRACT_DIR)], check=True)
else:
    print(f"Dataset already extracted at: {BASE_DIR}")

In [ ]:
# Keep all dataset and output paths in one place
IMG_DIR = BASE_DIR / "images"
ANNOT_ROOT = BASE_DIR / "Annotations"
ORIG_ANNOT_DIR = ANNOT_ROOT / "Original_Format"
COCO_ANNOT_DIR = ANNOT_ROOT / "COCO_compatible_format"
HF_ANNOT_DIR = ANNOT_ROOT / "layoutlm_HF_format"

WORK_DIR = Path("/kaggle/working/findocai")
MANIFEST_DIR = WORK_DIR / "manifests"
OCR_OUT_DIR = WORK_DIR / "ocr" / "doctr_default"
AUDIT_OUT_DIR = Path("/kaggle/working/evaluation_results")

for folder in (MANIFEST_DIR, OCR_OUT_DIR, AUDIT_OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dataset contents:", os.listdir(BASE_DIR))
print("Annotation formats:", os.listdir(ANNOT_ROOT))
print("Using device:", DEVICE)

## 2. Dataset & Annotation Inspection

Check the default CSV splits (`strat1`), confirm every image has a matching JSON annotation, and inspect the keys/structure of all three annotation formats shipped with FATURA2.

In [ ]:
# Default strat1 splits shipped with the dataset
strat1_train = pd.read_csv(BASE_DIR / "strat1_train.csv")
strat1_val = pd.read_csv(BASE_DIR / "strat1_dev.csv")
strat1_test = pd.read_csv(BASE_DIR / "strat1_test.csv")

print(f"strat1 shapes -> train: {strat1_train.shape}, val: {strat1_val.shape}, test: {strat1_test.shape}")
display(strat1_train.head())

In [ ]:
# Make sure image and annotation counts line up cleanly
img_files = sorted(p for p in IMG_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
annot_files = sorted(ORIG_ANNOT_DIR.glob("*.json"))

print(f"Images found: {len(img_files)} | Original_Format JSONs: {len(annot_files)}")

# Sanity check that no stray JSONs ended up inside the images folder
stray_jsons = list(IMG_DIR.glob("*.json"))
print(f"Stray JSONs in images dir: {len(stray_jsons)}")

In [ ]:
# Scan all Original_Format JSONs to list every unique annotation key in the dataset
all_fields = set()
for json_path in annot_files:
    with open(json_path, "r", encoding="utf-8") as f:
        all_fields.update(json.load(f).keys())

ALL_FIELDS = sorted(all_fields)
print(f"Total unique fields: {len(ALL_FIELDS)}")
print(ALL_FIELDS)

In [ ]:
# Inspect one sample invoice across Original, COCO, and LayoutLM HF formats
sample_name = "Template1_Instance0"

with open(ORIG_ANNOT_DIR / f"{sample_name}.json", "r", encoding="utf-8") as f:
    orig_sample = json.load(f)

print("Original_Format keys:", list(orig_sample.keys()))
print("Sample TOTAL field:", orig_sample.get("TOTAL"))

coco_sample_path = next(COCO_ANNOT_DIR.glob("*.json"))
with open(coco_sample_path, "r", encoding="utf-8") as f:
    coco_sample = json.load(f)

print("\nCOCO sample file:", coco_sample_path.name)
print("COCO keys:", list(coco_sample.keys()))
print("Image ID:", coco_sample.get("image_id"), "| Orig size:", coco_sample.get("orig_size"))
print("Labels count:", len(coco_sample.get("class_labels", [])), "| Boxes count:", len(coco_sample.get("boxes", [])))
print("First 3 boxes:", coco_sample.get("boxes", [])[:3])

hf_sample_path = glob.glob(str(HF_ANNOT_DIR / f"{sample_name}*"))[0]
with open(hf_sample_path, "r", encoding="utf-8") as f:
    hf_sample = json.load(f)

print("\nLayoutLM HF sample file:", Path(hf_sample_path).name)
print("HF preview:", {k: (v[:8] if isinstance(v, list) else v) for k, v in hf_sample.items()})

### Coordinate Orientation Check (Y-Axis Flip)

Bounding boxes in `Original_Format` come from PDF coordinates (origin at bottom-left), whereas image arrays place `(0, 0)` at the top-left. Plotting raw boxes next to `H - y` flipped boxes confirms whether we need to invert the Y-axis during label alignment.

In [ ]:
img = Image.open(IMG_DIR / f"{sample_name}.jpg")
W, H = img.size

fig, axes = plt.subplots(1, 2, figsize=(14, 10))
for ax, flip_y in zip(axes, [False, True]):
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Flipped Y (H - y)" if flip_y else "Raw Coordinates (as-is)")

    for field_name, val in orig_sample.items():
        if not isinstance(val, dict) or "bbox" not in val:
            continue
        (x1, y1), (x2, y2) = val["bbox"]
        if flip_y:
            y1, y2 = H - y1, H - y2

        rect = patches.Rectangle(
            (min(x1, x2), min(y1, y2)),
            abs(x2 - x1),
            abs(y2 - y1),
            fill=False,
            edgecolor="red",
            linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(min(x1, x2), min(y1, y2), field_name, color="blue", fontsize=6)

plt.tight_layout()
plt.show()

## 3. Single-Image OCR Sanity Check (`docTR`)

Run `docTR` on `Template1_Instance0.jpg` to check raw detection boxes, inspect the synthesized page layout, and verify whether the ground-truth `TOTAL` text appears in the extracted OCR output.

In [ ]:
# assume_straight_pages=False lets docTR handle slight rotation/skew if present
ocr_model = ocr_predictor(pretrained=True, assume_straight_pages=False).to(DEVICE)
ocr_model.eval()

sample_doc = DocumentFile.from_images(str(IMG_DIR / f"{sample_name}.jpg"))
with torch.inference_mode():
    sample_result = ocr_model(sample_doc)

sample_result.show()

In [ ]:
# Reconstruct the page from OCR predictions to visually spot missed words or garbled reading order
synthetic_pages = sample_result.synthesize()
plt.figure(figsize=(8, 10))
plt.imshow(synthetic_pages[0])
plt.axis("off")
plt.show()

In [ ]:
def normalize_text(text):
    # Lowercase and collapse whitespace while keeping punctuation and currency symbols intact
    return re.sub(r"\s+", " ", str(text).lower()).strip()

def get_ocr_text(ocr_result):
    words = [
        word.value
        for page in ocr_result.pages
        for block in page.blocks
        for line in block.lines
        for word in line.words
    ]
    return " ".join(words)

gt_total = orig_sample["TOTAL"]["text"]
ocr_text = get_ocr_text(sample_result)

clean_gt = normalize_text(gt_total)
clean_ocr = normalize_text(ocr_text)

print("Normalized GT TOTAL:", repr(clean_gt))
print("Status:", "MATCH" if clean_gt in clean_ocr else "MISMATCH")

## 4. OCR vs. Ground Truth Audit

To see where `docTR` struggles across the dataset, we compare normalized ground-truth field strings against the full page OCR output for all 10,000 invoices.

Because running this over 10k images takes a while, it is guarded by `RUN_MISMATCH_AUDIT = False` by default. The next cell loads the precomputed `fatura_mismatches.csv` file and summarizes mismatches per field.

In [ ]:
# LOGO is purely visual, so exclude it from text-matching checks
NON_TEXT_FIELDS = {"LOGO"}

# Keys that carry geometry/metadata rather than actual invoice text
IGNORED_META_KEYS = {
    "bbox", "box", "coordinates", "points",
    "vertices", "confidence", "score", "id", "page",
}

def is_missing(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and not value.strip():
        return True
    return False

def extract_field_texts(value):
    """Pull string/numeric text out of nested annotation entries while skipping bbox metadata."""
    extracted = []
    if is_missing(value):
        return extracted

    if isinstance(value, (str, int, float)) and not isinstance(value, bool):
        return [str(value).strip()]

    if isinstance(value, dict):
        # Prefer explicit text keys before falling back to scanning nested items
        preferred = [k for k in ("text", "value", "content") if k in value]
        if preferred:
            for k in preferred:
                extracted.extend(extract_field_texts(value[k]))
            return extracted

        for k, v in value.items():
            if k.lower() not in IGNORED_META_KEYS:
                extracted.extend(extract_field_texts(v))
    elif isinstance(value, list):
        for item in value:
            extracted.extend(extract_field_texts(item))

    return extracted

In [ ]:
RUN_MISMATCH_AUDIT = False  # Flip to True only if you need to regenerate fatura_mismatches.csv

if RUN_MISMATCH_AUDIT:
    annot_map = {p.stem: p for p in annot_files}
    mismatch_records = []
    processing_errors = []
    processed_count = 0

    for image_path in tqdm(img_files, desc="Auditing OCR vs GT"):
        try:
            annot_path = annot_map.get(image_path.stem)
            if annot_path is None:
                processing_errors.append({"image": image_path.name, "error": "Missing annotation JSON"})
                continue

            with open(annot_path, "r", encoding="utf-8") as f:
                annotation = json.load(f)
            if not isinstance(annotation, dict):
                raise ValueError("Top-level annotation is not a dict")

            doc = DocumentFile.from_images(str(image_path))
            with torch.inference_mode():
                ocr_res = ocr_model(doc)
            clean_ocr = normalize_text(get_ocr_text(ocr_res))

            for field in ALL_FIELDS:
                if field in NON_TEXT_FIELDS or field not in annotation:
                    continue

                gt_values = extract_field_texts(annotation[field])
                if not gt_values:
                    continue

                missing = [
                    val for val in gt_values
                    if normalize_text(val) and normalize_text(val) not in clean_ocr
                ]
                if missing:
                    mismatch_records.append({
                        "image": image_path.name,
                        "annotation_file": annot_path.name,
                        "field": field,
                        "ground_truth": " | ".join(gt_values),
                        "missing_values": " | ".join(missing),
                        "status": "MISMATCH",
                    })

            processed_count += 1
        except Exception as e:
            processing_errors.append({"image": image_path.name, "error": str(e)})

    audit_df = pd.DataFrame(mismatch_records)
    audit_df.to_csv(AUDIT_OUT_DIR / "fatura_mismatches.csv", index=False)
    print(f"Done. Processed: {processed_count} | Mismatches: {len(mismatch_records)} | Errors: {len(processing_errors)}")
else:
    print("Skipping full OCR audit (RUN_MISMATCH_AUDIT=False). Loading precomputed CSV next.")

In [ ]:
# Load precomputed mismatch CSV (prefers newly generated file, then Kaggle input mount, then kagglehub)
kaggle_csv = Path("/kaggle/input/datasets/satyamkarn100/fatura-mismatch-df/fatura_mismatches.csv")
local_audit_csv = AUDIT_OUT_DIR / "fatura_mismatches.csv"

if local_audit_csv.exists():
    csv_path = local_audit_csv
elif kaggle_csv.exists():
    csv_path = kaggle_csv
else:
    import kagglehub
    # Uncomment if the Kaggle dataset is private:
    # kagglehub.login()
    dl_dir = Path(kagglehub.dataset_download("satyamkarn100/fatura-mismatch-df"))
    csv_path = dl_dir / "fatura_mismatches.csv"

mismatch_df = pd.read_csv(csv_path)
print(f"Loaded {len(mismatch_df):,} mismatch rows from {csv_path}")
display(mismatch_df.head(10))

field_summary = (
    mismatch_df["field"]
    .value_counts()
    .rename_axis("Field")
    .reset_index(name="Mismatches")
)
display(field_summary)

## 5. Leakage-Free Template Split & Manifest Freeze

The default `strat1` split shares the same invoice templates across train, dev, and test. To test actual generalization to unseen invoice layouts, we create a template-disjoint **32 / 8 / 10** split across all 50 templates:
- **Train:** 32 templates (6,400 images), with every 10th instance flagged as `is_inlayout_holdout` (640 images) for tracking in-layout vs. unseen-layout performance.
- **Validation:** 8 templates (1,600 images).
- **Test:** 10 templates (2,000 images), locked with a SHA-256 hash so the test set never drifts.

In [ ]:
jpg_names = sorted(f for f in os.listdir(IMG_DIR) if f.endswith(".jpg"))

manifest = pd.DataFrame({"img_path": jpg_names})
manifest["annot_path"] = manifest["img_path"].str.replace(".jpg", ".json", regex=False)
manifest["template_id"] = manifest["img_path"].str.extract(r"Template(\d+)_")[0].astype(int)
manifest["instance_id"] = manifest["img_path"].str.extract(r"Instance(\d+)")[0].astype(int)

# Deterministic shuffle (seed=42) over the 50 unique templates
templates = sorted(int(t) for t in manifest["template_id"].unique())
assert len(templates) == 50, f"Expected 50 templates, found {len(templates)}"

random.seed(42)
random.shuffle(templates)

train_templates = set(templates[:32])
val_templates = set(templates[32:40])
test_templates = set(templates[40:])

manifest["split"] = manifest["template_id"].map(
    lambda t: "train" if t in train_templates else ("val" if t in val_templates else "test")
)

# Hold out 10% of training instances for in-layout evaluation
manifest["is_inlayout_holdout"] = (manifest["split"] == "train") & (manifest["instance_id"] % 10 == 0)

# Verify zero template overlap across splits
split_sets = {s: set(g["template_id"]) for s, g in manifest.groupby("split")}
assert split_sets["train"].isdisjoint(split_sets["val"])
assert split_sets["train"].isdisjoint(split_sets["test"])
assert split_sets["val"].isdisjoint(split_sets["test"])

print("Split counts:", manifest["split"].value_counts().to_dict())
print(
    "Templates per split ->",
    f"train: {len(split_sets['train'])}, val: {len(split_sets['val'])}, test: {len(split_sets['test'])}",
)
print("In-layout holdout images:", int(manifest["is_inlayout_holdout"].sum()))
print("Frozen test templates:", sorted(test_templates))

In [ ]:
# Write manifest and freeze the sorted test image list via SHA-256
manifest_path = MANIFEST_DIR / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

test_files_joined = "\n".join(sorted(manifest.loc[manifest["split"] == "test", "img_path"]))
test_hash = hashlib.sha256(test_files_joined.encode("utf-8")).hexdigest()

(MANIFEST_DIR / "test_freeze_sha256.txt").write_text(test_hash, encoding="utf-8")
print(f"Saved manifest to {manifest_path}")
print(f"Frozen test SHA-256: {test_hash[:16]}...")

## 6. Sharded Full-Dataset OCR Extraction & Hugging Face Sync

Run `docTR` (default straight-page mode for faster inference on clean digital invoices) in batches across all 10,000 invoices, saving normalized word boxes and line IDs to `.jsonl` shards.

Each shard is written atomically via a `.tmp` file and pushed to `Satyamkarn100/FinDoc-AI` on Hugging Face Hub. If the runtime disconnects, re-running this stage downloads existing remote shards and resumes only on unprocessed images.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

REPO_ID = "Satyamkarn100/FinDoc-AI"
HF_OCR_DIR = "ocr/doctr_default"

# Read HF_TOKEN from Kaggle secrets when on Kaggle, otherwise fall back to environment variable
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True)
print("Connected to Hugging Face as:", api.whoami()["name"])

# Push manifests folder so downstream training notebooks pull the exact same splits
api.upload_folder(
    folder_path=str(MANIFEST_DIR),
    path_in_repo="manifests",
    repo_id=REPO_ID,
    repo_type="dataset",
)

In [ ]:
# Sync any existing shards from HF Hub so we don't re-OCR completed images
remote_files = set(api.list_repo_files(repo_id=REPO_ID, repo_type="dataset"))
remote_shards = sorted(
    f for f in remote_files
    if f.startswith(f"{HF_OCR_DIR}/") and f.endswith(".jsonl")
)
print(f"Remote shards found in '{HF_OCR_DIR}': {len(remote_shards)}")

if remote_shards:
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        local_dir=str(WORK_DIR),
        allow_patterns=[f"{HF_OCR_DIR}/*.jsonl"],
        token=HF_TOKEN,
    )

# Build the set of already-extracted filenames across all local shards
done_files = set()
for shard_path in sorted(OCR_OUT_DIR.glob("shard_*.jsonl")):
    with open(shard_path, "r", encoding="utf-8") as fh:
        for line in fh:
            done_files.add(json.loads(line)["filename"])

LIMIT = None  # Set to a small int (e.g. 50) for a quick smoke test
todo_images = sorted(p for p in IMG_DIR.glob("*.jpg") if p.name not in done_files)
if LIMIT is not None:
    todo_images = todo_images[:LIMIT]

print(f"Already done: {len(done_files)} | Remaining to process: {len(todo_images)}")

In [ ]:
# Default docTR predictor (straight pages) runs significantly faster over 10k synthetic invoices
batch_predictor = ocr_predictor(pretrained=True).to(DEVICE)
batch_predictor.eval()

def extract_page_words(page_dict):
    """Flatten docTR exported page dict into [text, conf, x1, y1, x2, y2, line_id] lists."""
    words = []
    line_id = 0
    for block in page_dict["blocks"]:
        for line in block["lines"]:
            for w in line["words"]:
                (x1, y1), (x2, y2) = w["geometry"]
                words.append([
                    w["value"],
                    round(w["confidence"], 4),
                    round(x1, 5), round(y1, 5),
                    round(x2, 5), round(y2, 5),
                    line_id,
                ])
            line_id += 1
    return words

def flush_shard(records_buffer):
    """Atomically save buffered OCR records to the next shard file and upload it to HF."""
    if not records_buffer:
        return None

    shard_idx = len(list(OCR_OUT_DIR.glob("shard_*.jsonl")))
    shard_name = f"shard_{shard_idx:04d}.jsonl"
    tmp_path = OCR_OUT_DIR / f"{shard_name}.tmp"
    final_path = OCR_OUT_DIR / shard_name

    with open(tmp_path, "w", encoding="utf-8") as fh:
        for rec in records_buffer:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

    tmp_path.replace(final_path)
    records_buffer.clear()

    api.upload_file(
        path_or_fileobj=str(final_path),
        path_in_repo=f"{HF_OCR_DIR}/{shard_name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Upload {shard_name}",
    )
    print(f"Saved & uploaded {shard_name}")
    return final_path

In [ ]:
BATCH_SIZE = 8
SHARD_SIZE = 2000  # Periodic flush so a session timeout never wipes out progress

buffer = []
errors = []
last_sample = None
t0 = time.time()

for i in tqdm(range(0, len(todo_images), BATCH_SIZE), desc="Extracting OCR"):
    batch = todo_images[i : i + BATCH_SIZE]
    try:
        docs = DocumentFile.from_images([str(p) for p in batch])
        with torch.inference_mode():
            exported = batch_predictor(docs).export()

        for img_path, img_arr, page in zip(batch, docs, exported["pages"]):
            h, w = img_arr.shape[:2]
            record = {
                "filename": img_path.name,
                "width": w,
                "height": h,
                "words": extract_page_words(page),
            }
            buffer.append(record)
            last_sample = record

        if len(buffer) >= SHARD_SIZE:
            flush_shard(buffer)

    except Exception as e:
        err_msg = f"{batch[0].name}: {e}"
        errors.append(err_msg)
        print("Batch error:", err_msg)

# Flush remaining buffered records at the end of the run
if buffer:
    flush_shard(buffer)

elapsed = (time.time() - t0) / 60
print(f"Finished in {elapsed:.1f} min | Errors: {len(errors)}")

### Post-Run Verification

Spot-check one extracted record and verify total image coverage across all local shards.

In [ ]:
if last_sample is not None:
    print("Sample file:", last_sample["filename"])
    print("Dimensions:", f"{last_sample['width']} x {last_sample['height']}")
    print("First 2 words (text, conf, x1, y1, x2, y2, line_id):")
    print(last_sample["words"][:2])

all_shard_files = sorted(OCR_OUT_DIR.glob("shard_*.jsonl"))
total_records = 0
for sp in all_shard_files:
    with open(sp, "r", encoding="utf-8") as fh:
        names = [json.loads(line)["filename"] for line in fh]
    total_records += len(names)
    print(f"{sp.name}: {len(names)} images ({names[0]} .. {names[-1]})")

print(f"\nTotal OCR records across {len(all_shard_files)} shard(s): {total_records}")